## simple_triton example

This notebook illustrates how to use the simple_triton package to perform inference on tiles from a whole-slide image using a Triton inference server.

Notes for running:
- Run this notebook in a container with `--network=host` so that it can reach the Triton container
- Mount the EfficientNetV2S.tensorflow savedmodel directory to the triton container
- Load the model (below)

In [1]:
# install large_image with tile sources
!apt update
!apt install -y python3-openslide openslide-tools
!pip install ../../histomics_stream 'large_image[tiff,openslide]' \
  scikit_image --find-links https://girder.github.io/large_image_wheels

# install simple_triton
!pip install ../../simple_triton

# install mil
!pip install ../../mil

Hit:1 http://archive.ubuntu.com/ubuntu focal InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2004/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu focal-updates InRelease                 
Hit:4 http://archive.ubuntu.com/ubuntu focal-backports InRelease             
Ign:5 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  InRelease
Hit:6 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  Release
Hit:7 http://security.ubuntu.com/ubuntu focal-security InRelease
Reading package lists... Done
Building dependency tree       
Reading state information... Done
124 packages can be upgraded. Run 'apt list --upgradable' to see them.
Reading package lists... Done
Building dependency tree       
Reading state information... Done
openslide-tools is already the newest version (3.4.1+dfsg-4).
python3-openslide is already the newest version (1.1.1-4.1build1).
0 upgraded, 0 newly installed, 0

  Created wheel for histomics-stream: filename=histomics_stream-2.3.0-py3-none-any.whl size=25526 sha256=7bad69a00fd09653e5318125350945b41e43cfca3254d8eac5d9b34f560457e8
  Stored in directory: /tmp/pip-ephem-wheel-cache-p_j8dmue/wheels/4d/92/92/a22ebaf41cdb2b39fb056793fde0f5347a037c4cd09293a37a
Successfully built histomics-stream
  Attempting uninstall: histomics-stream
    Found existing installation: histomics-stream 2.3.0
    Uninstalling histomics-stream-2.3.0:
      Successfully uninstalled histomics-stream-2.3.0
You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.
Processing /tf/notebooks/simple_triton
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
    Preparing wheel metadata ... done
  Created wheel for simple-triton: filename=simple_triton-0.1.dev168+g28c9b18.d20230320-py3-none-any.whl size=16193 sha256=3e77532e529f5438683af33aade9759679f9eafbd933fa2944

  Attempting uninstall: simple-triton
    Found existing installation: simple-triton 0.1.dev168+g28c9b18.d20230320
    Uninstalling simple-triton-0.1.dev168+g28c9b18.d20230320:
      Successfully uninstalled simple-triton-0.1.dev168+g28c9b18.d20230320
You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.
Processing /tf/notebooks/mil
  Created wheel for mil: filename=mil-0.0.1-py3-none-any.whl size=31277 sha256=93aa071831d58abcedadc2ae207814f41f6be7e5f716b896468ae2cc9eeabc61
  Stored in directory: /tmp/pip-ephem-wheel-cache-vrlw_keh/wheels/66/94/a3/a779f7ce14d952d91908932ab804b3c945d66a54c24e4eb96f
Successfully built mil
  Attempting uninstall: mil
    Found existing installation: mil 0.0.1
    Uninstalling mil-0.0.1:
      Successfully uninstalled mil-0.0.1
You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


## Create a histomics stream study

Parameters in this cell are for reading from the whole-slide image (magnification, tile size, tile overlap, mask file).

In [2]:
from mil.io.utils import study

# slide parameters
batch = 64
magnification = 20
tile = 224
overlap = 0
chunk = 224
mask_threshold=0.5
wsi_path = "/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.svs"
mask_path = "/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.mask.png"

# create a histomics-stream study from a wsi/mask pair
hs_study = study((wsi_path, mask_path),
                 t=(tile, tile),
                 chunk=(tile, tile),
                 target=20,
                 source="exact")

## Create and load model

The function `feature_extractor` can be used to create feature extraction models in the model repository. Note - this cell will take time as the model is downloaded, saved, and loaded into triton. 

Here we generate a model, load the model into triton, and verify that the model state is "READY".

Parameters in this stage include the inference server (address), the model (model name, maximum batch size).

In [3]:
import json
from google.protobuf.json_format import MessageToDict
import numpy as np
from simple_triton.feature_extraction import feature_extractor
from simple_triton.model import model_config
import tritonclient.grpc as grpcclient

# triton parameters
url = "localhost:8001"  # url for grpc access to tirton server
keras_name = "EfficientNetV2S"
model_name = f"{keras_name}.tensorflow"  # set model name

# create the model and capture output dimensionality
try:
    dimension_output = feature_extractor("/tf/notebooks/models",
                                         keras_name,
                                         model_name,
                                         t=(tile, tile),
                                         pool="avg")
except:
    print(f"Model {model_name} already exists.")

# create triton client
client = grpcclient.InferenceServerClient(url=url, verbose=True)

# load tensorflow model with larger batch size
config = {'maxBatchSize': 256}
client.load_model(model_name, config=json.dumps(config))

# check readiness
client.get_model_repository_index()

# deleting the client in main prevents conflicts with child process clients
del client

Imported version of grpc is 1.46.1. There is a memory leak in certain Python GRPC versions (1.43.0 to be specific). Please use versions <1.43.0 or >=1.51.1 to avoid leaks (see https://github.com/grpc/grpc/issues/28513).


Model EfficientNetV2S.tensorflow already exists.
load_model, metadata ()
override files omitted:
model_name: "EfficientNetV2S.tensorflow"
parameters {
  key: "config"
  value {
    string_param: "{\"maxBatchSize\": 256}"
  }
}

Loaded model 'EfficientNetV2S.tensorflow'
get_model_repository_index, metadata ()

models {
  name: ".ipynb_checkpoints"
}
models {
  name: "EfficientNetV2L.tensorflow"
  version: "1"
  state: "READY"
}
models {
  name: "EfficientNetV2S.onnx"
}
models {
  name: "EfficientNetV2S.tensorflow"
  version: "1"
  state: "READY"
}
models {
  name: "EfficientNetV2S_TRT_FP16.tensorflow"
}
models {
  name: "EfficientNetV2S_TRT_FP32.tensorflow"
}



## Run the inference

Parameters here include the number of tiles per batch, the number of workers, and the maximum number of pending inferences per worker.

In [4]:
from simple_triton.feature_extraction import histomics_stream_inference
from simple_triton.submitter import analyze
import time

# inference parameters
batch = 64
limit = 10  # limit on number of pending requests per worker
workers = 32  # total number of Submitter workers
verbose = True  # set verbose as False

# start timer
start = time.time()

# inference
features, results, times = histomics_stream_inference(hs_study, 
                                                      model_name, 
                                                      url="localhost:8001", 
                                                      batch=batch, 
                                                      workers=workers,
                                                      limit=limit)

# display elapsed time
print(f"Total elapsed time: {time.time()-start}")

# analyze performance
analyze(times)

Total elapsed time: 8.299997329711914
                             median    min    max
-------------------------  --------  -----  -----
total (sec)                    3.80   0.63   7.36
qin (% total)                 68.22  30.72  89.20
qout (% total)                 0.10   0.03   0.47
in-process (% total)          31.67  10.58  69.21
completion (% in-process)     75.24  25.34  98.22
retrieval (% in-process)      14.59   0.00  66.23
other (% in-process)           5.53   1.49  24.27
